In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns 
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.DataStructs.cDataStructs import ExplicitBitVect
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.model_selection import train_test_split
from tanimoto import FastTanimotoKernel

In [2]:
class ChemCalculator(FastTanimotoKernel):
    def __init__(self, data_set):
        super().__init__()
        self.data_set = data_set
        self.fingerprint = []
        self.read_set = pd.read_csv(self.data_set)

    def show_data(self):
        # Show the database
        print(self.read_set)

    def select_data(self, smiles, property, number_of_data):
        # Select the features choosing the size 
        self.read_set = self.read_set[0:number_of_data]
        self.data_smiles = self.read_set[smiles]
        self.data_property = self.read_set[property]

        self.data_smiles = self.data_smiles.tolist()
        self.data_property = self.data_property.tolist()
        return self.data_smiles, self.data_property
    
    def convert_smiles_to_fingerprint(self, data_smiles, radius=2, nBits=2048):
        # Smiles list convert in to a fingerprint vector
        self.molecul = Chem.MolFromSmiles(data_smiles)
        self.fp = AllChem.GetMorganFingerprintAsBitVect(self.molecul, radius=radius, nBits=nBits)
        return self.fp
    
    def matrix_fingerprints(self):
        # Form with fingerprint vectors a matrix
        self.fingerprint = []
        if self.data_smiles:
            self.fingerprint = [self.convert_smiles_to_fingerprint(smile) for smile in self.data_smiles]
        else:
            print("No data SMILES selected. Please use select_data method first.")
    
    def get_fingerprints(self):
        # Show the fingerprint matrix
        if not self.fingerprint:
            print("Fingerprint matrix is empty. Please use matrix_fingerprints method first.")
        return self.fingerprint

    def prepare_data(self):
        # Select the percentage of train data and define the X_train, y_train, X_test and y_test
        if self.fingerprint and self.data_property:
            self.y = self.data_property
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(self.fingerprint, self.y, test_size=0.2, random_state=42)

            return self.X_train, self.X_test, self.y_train, self.y_test
        
    def gp_train(self):
        # Train the model
        if not self.fingerprint or not self.data_property:
            print("Fingerprint matrix or property data is missing. Please ensure both are available.")
            return None
        self.kernel = FastTanimotoKernel()
        self.gp = GaussianProcessRegressor(kernel=self.kernel,alpha=1e-6, normalize_y=True)
        self.gp.fit(self.X_train, self.y_train)
    
    def gp_input_predict(self, test_smiles):
        # Make a prediction introducing a smiles list
        self.test_fp = [self.convert_smiles_to_fingerprint(smile) for smile in test_smiles]
        self.X_test = np.array(self.test_fp, dtype=object)
        self.y_pred, self.y_std = self.gp.predict(self.X_test, return_std=True)
        print("Predicted values:", self.y_pred)
        print("Uncertainity:",self.y_std)
    
    def gp_predict(self):
        # Predict a set of know values
        self.y_pred, self.y_std = self.gp.predict(self.X_test, return_std=True)
        print("Predicted values:", self.y_pred)
        print("Uncertainity:",self.y_std)

    def get_predictions(self):
        return self.y_pred
    
    def get_uncertainty(self):
        return self.y_std

In [3]:
propCal = ChemCalculator("qm9.csv")

In [4]:
propCal.select_data(smiles="smiles", property="gap", number_of_data=9000)
propCal.matrix_fingerprints()
propCal.prepare_data()
propCal.gp_train()
propCal.gp_predict()

Predicted values: [0.25520894 0.21994155 0.27082845 ... 0.21522465 0.2677386  0.33887849]
Uncertainity: [0.02981757 0.02794816 0.0245608  ... 0.02775932 0.03176941 0.02502802]
